Representational Similarity Analysis (RSA): how does the spatial pattern of
body maps across the emotions relate to one another / what is their overall structure?

Embody body-map RDM
- For each participant, rank the pixels of each emotion map and correlate every emotion pair (Spearman).
- Dissimilarity = 1 − ρ (0 = identical pattern, 1 = unrelated, 2 = inverse).
- Subject RDMs are averaged into one group RDM, then plotted as a heatmap and a dendrogram.
- Input: one .npz per participant with resmat. Output: Excel sheet for numerical RDM results, RDM PNG, dendrogram PNG.

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from tqdm.auto import tqdm

# ---------------------------------------------------------------------------
# Per-subject RDM
# ---------------------------------------------------------------------------

def subject_body_rdm(maps):
   flat = maps.reshape(-1, maps.shape[-1])
   flat = flat[np.isfinite(flat).all(axis=1)]
   with np.errstate(invalid="ignore", divide="ignore"):
       #Rank, then Pearson, equals Spearman. Pixels with a NaN in any emotion are dropped (listwise). A constant or blank map gives NaN, which nanmean skips later.
       return 1 - np.corrcoef(rankdata(flat, axis=0), rowvar=False)

# ---------------------------------------------------------------------------
# Streaming group RDM: loads ONE subject at a time, never stacks 2000 of them
# ---------------------------------------------------------------------------
def group_body_rdm_from_npz(npz_folder, N, keep_idx=None):
    """
    Parameters
    ----------
    npz_folder : str or Path
        Folder containing one *.npz file per subject, each with a
        'resmat' array of shape (600, 235, N).
    N : int
        emotion count before dropping

    Returns
    -------
    rdm : ndarray, shape (N, N)
    n_used : int
        Number of subjects actually included (after skipping bad files).
    skipped: int
        Number of bad files skipped.
    """
    npz_folder = Path(npz_folder)
    npz_files = sorted(npz_folder.glob("*.npz"))

    if not npz_files:
        raise FileNotFoundError(f"No files matching '*.npz' found in:\n{npz_folder}")

    subject_rdms = []
    skipped=0

    iterator = tqdm(npz_files, desc="Computing subject RDMs", unit="subj")

    for file_path in iterator:
        with np.load(file_path) as saved:
            if "resmat" not in saved:
                print(f"Skipping {file_path.name}: no 'resmat' variable")
                skipped += 1
                iterator.set_postfix(skipped=skipped)
                continue
            resmat = saved["resmat"].astype(np.float32)

        if resmat.shape != (600, 235, N):
            print(f"Skipping {file_path.name}: shape {resmat.shape}, expected {(600, 235, N)}")
            skipped += 1
            iterator.set_postfix(skipped=skipped)
            continue

        if keep_idx is not None:
            resmat = resmat[:, :, keep_idx]   # drop unwanted emotions

        subject_rdms.append(subject_body_rdm(resmat))

    with np.errstate(invalid="ignore"):
        rdm = np.nanmean(subject_rdms, axis=0)

    np.fill_diagonal(rdm, 0.0)
    return rdm, len(subject_rdms), skipped

# ---------------------------------------------------------------------------
# Plotting helpers
# ---------------------------------------------------------------------------

def plot_rdm(rdm, title, output_path,labels,y_label):
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad("white")
    masked = np.ma.masked_array(rdm, np.triu(np.ones_like(rdm, bool), k=1))
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(masked, cmap=cmap)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label=f"Dissimilarity: {y_label}")
    fig.tight_layout()
    fig.savefig(output_path, dpi=120)
    plt.close(fig)

def reorder_for_display(rdm, labels, display_order):
    """
    Reorder an RDM's rows/columns and labels for plotting only.
    Does not change the underlying RDM computation or the original
    LABELS order used elsewhere in the pipeline.
    """
    idx = [labels.index(lab) for lab in display_order]
    return rdm[np.ix_(idx, idx)], display_order

def move_leaf_right(Z, leaf_idx):
    """Swap children so leaf `leaf_idx` ends up as the right-most leaf."""
    Z = Z.copy()
    n = Z.shape[0] + 1                       # number of leaves
    members = {i: {i} for i in range(n)}     # leaves under each node
    for i in range(Z.shape[0]):
        a, b = int(Z[i, 0]), int(Z[i, 1])
        members[n + i] = members[a] | members[b]
        if leaf_idx in members[a]:           # target is in the left branch -> flip
            Z[i, 0], Z[i, 1] = Z[i, 1], Z[i, 0]
    return Z

def plot_dendrogram(rdm, labels, title, output_path, method, y_label,right_label=None):

    condensed = squareform(rdm, checks=False)
    Z = linkage(condensed, method=method)

    if right_label is not None:
        Z = move_leaf_right(Z, labels.index(right_label))

    fig, ax = plt.subplots(figsize=(8, 5))
    dendrogram(Z, labels=labels, ax=ax, leaf_rotation=90, leaf_font_size=9)
    ax.set_title(title)
    ax.set_ylabel(f"Dissimilarity: {y_label}")
    ax.set_ylim()
    fig.tight_layout()
    fig.savefig(output_path, dpi=120)
    plt.close(fig)

Edit/Run below

In [10]:
ALL_LABELS = [
    'Neutral', 'Fear', 'Anger', 'Disgust', 'Sadness', 'Happiness',
    'Lonely', 'Anxiety', 'Love', 'Depression', 'Pride', 'Shame',
    'Jealousy', 'Stress', 'Migraine', 'Nausea',]
#Order must match the 3rd axis of resmat

DROP = {}

keep_idx = [i for i, lab in enumerate(ALL_LABELS) if lab not in DROP]
LABELS = [ALL_LABELS[i] for i in keep_idx]

npz_folder = Path("/Users/hagrid/Downloads/preprocessed_complete_prolific")
output_folder = Path("/Users/hagrid/Desktop/ML/Embody/Results/Test")
output_folder.mkdir(parents=True, exist_ok=True)

run_timestamp = datetime.now().strftime("%Y-%m-%d %H-%M")
excel_path = output_folder / f"RDMS results {run_timestamp}.xlsx"

rdm, n_subjects_used,skipped = group_body_rdm_from_npz(
    npz_folder, N=len(ALL_LABELS), keep_idx=keep_idx)

print(f"Group RDM built from {n_subjects_used} subjects (skipped {skipped} subjects), shape {rdm.shape}")  # (14, 14)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    pd.DataFrame(rdm, index=LABELS, columns=LABELS).to_excel(writer, sheet_name="Body maps")

plot_rdm(rdm, "Emotion Body Maps RDM", output_folder / f"Rdm body maps {run_timestamp}.png", LABELS,"1 − Spearman's ρ")

plot_dendrogram(rdm, LABELS, "Emotion body maps: hierarchical clustering",output_folder / f"Dendrogram body maps {run_timestamp}.png",method="average", y_label="1 − Spearman's ρ",)
#Average linkage (UPGMA), because 1 − ρ is not Euclidean, so Ward doesn’t apply.

Computing subject RDMs: 100%|██████████| 2427/2427 [02:40<00:00, 15.15subj/s]


Group RDM built from 2427 subjects (skipped 0 subjects), shape (14, 14)
